In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import statsmodels.stats.multitest as multitest
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. Global Page and Font Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# ---------------------------------------------------------
# Mathtext configuration for independent styling:
# - \mathit{} maps to regular Italic (for r and q)
# - \mathtt{} is remapped to Bold Italic (for Genus)
# - \mathbf{} maps to regular Bold (for spp.)
# ---------------------------------------------------------
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'
plt.rcParams['mathtext.tt'] = 'Arial:italic:bold'

def clean_and_convert(vals):
    s_vals = pd.Series(vals).astype(str).str.strip().replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', ''], np.nan)
    return pd.to_numeric(s_vals, errors='coerce')

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
df_ace = pd.read_csv('Acetate(mM).csv')
df_but = pd.read_csv('Butyrate(mM).csv')
donor_cols = [c for c in df_ace.columns if c.startswith('HS-')]

ctrl_ace = clean_and_convert(df_ace[df_ace.iloc[:, 0].astype(str).isin(['0', '000'])][donor_cols].iloc[0].values)
inul_ace = clean_and_convert(df_ace[df_ace['KULFFI'] == 'Inulin'][donor_cols].iloc[0].values)
delta_ace = inul_ace - ctrl_ace
ctrl_but = clean_and_convert(df_but[df_but.iloc[:, 0].astype(str).isin(['0', '000'])][donor_cols].iloc[0].values)
inul_but = clean_and_convert(df_but[df_but['KULFFI'] == 'Inulin'][donor_cols].iloc[0].values)
delta_but = inul_but - ctrl_but

df_16s = pd.read_csv('(KULFFI)_16S_Genus_level.csv')
bac_cols = [c for c in df_16s.columns if str(c).startswith('d__Bacteria')]
bifi_col = [c for c in df_16s.columns if 'g__Bifidobacterium' in c][0]
fae_col = [c for c in df_16s.columns if 'g__Faecalibacterium' in c][0]

donor_16s = {}
for cluster in df_16s['Cluster'].unique():
    df_cluster = df_16s[df_16s['Cluster'] == cluster]
    row_fec = df_cluster[df_cluster['Feature'] == 'FEC']
    if row_fec.empty:
        continue
    total_fec = row_fec[bac_cols].sum(axis=1).values[0]
    bifi_fec = (row_fec[bifi_col].values[0] / total_fec) * 100 if total_fec > 0 else np.nan
    fae_fec = (row_fec[fae_col].values[0] / total_fec) * 100 if total_fec > 0 else np.nan
    row_ctrl = df_cluster[df_cluster['Feature'] == 'CUL']
    row_inul = df_cluster[df_cluster['Feature'] == 'Inulin']

    total_ctrl = row_ctrl[bac_cols].sum(axis=1).values[0] if not row_ctrl.empty else 0
    total_inul = row_inul[bac_cols].sum(axis=1).values[0] if not row_inul.empty else 0

    bifi_ctrl = (row_ctrl[bifi_col].values[0] / total_ctrl) * 100 if total_ctrl > 0 else np.nan
    fae_ctrl = (row_ctrl[fae_col].values[0] / total_ctrl) * 100 if total_ctrl > 0 else np.nan
    bifi_inul = (row_inul[bifi_col].values[0] / total_inul) * 100 if total_inul > 0 else np.nan
    fae_inul = (row_inul[fae_col].values[0] / total_inul) * 100 if total_inul > 0 else np.nan

    donor_16s[cluster] = {
        'bifi_fec': bifi_fec, 'fae_fec': fae_fec,
        'bifi_ctrl': bifi_ctrl, 'bifi_inul': bifi_inul, 'bifi_delta': bifi_inul - bifi_ctrl,
        'fae_ctrl': fae_ctrl, 'fae_inul': fae_inul, 'fae_delta': fae_inul - fae_ctrl
    }

data = []
for i, donor in enumerate(donor_cols):
    if donor in donor_16s:
        d_info = donor_16s[donor]
        data.append({
            'Delta_Acetate': delta_ace.iloc[i], 'Delta_Butyrate': delta_but.iloc[i],
            'Bifi_Init': d_info['bifi_fec'], 'Bifi_Post': d_info['bifi_inul'], 'Bifi_Delta': d_info['bifi_delta'],
            'Fae_Init': d_info['fae_fec'], 'Fae_Post': d_info['fae_inul'], 'Fae_Delta': d_info['fae_delta']
        })
df_final = pd.DataFrame(data)

# ==========================================
# 3. Statistical Testing (FDR Correction)
# ==========================================
plot_configs = [
    ('Bifi_Init', r'Initial $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', r'$\Delta$ Acetate (mM)'),
    ('Bifi_Post', r'Post $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', ''),
    ('Bifi_Delta', r'$\Delta$ $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', ''),
    ('Bifi_Init', r'Initial $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', r'$\Delta$ Butyrate (mM)'),
    ('Bifi_Post', r'Post $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', ''),
    ('Bifi_Delta', r'$\Delta$ $\mathtt{Bifidobacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', ''),
    ('Fae_Init', r'Initial $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', r'$\Delta$ Acetate (mM)'),
    ('Fae_Post', r'Post $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', ''),
    ('Fae_Delta', r'$\Delta$ $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Acetate', ''),
    ('Fae_Init', r'Initial $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', r'$\Delta$ Butyrate (mM)'),
    ('Fae_Post', r'Post $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', ''),
    ('Fae_Delta', r'$\Delta$ $\mathtt{Faecalibacterium}$ $\mathbf{spp.}$ (%)', 'Delta_Butyrate', '')
]

p_values = []
valid_data_subsets = []
for x_col, _, y_col, _ in plot_configs:
    df_sub = df_final.dropna(subset=[x_col, y_col])
    x_data, y_data = df_sub[x_col], df_sub[y_col]
    r, p = pearsonr(x_data, y_data)
    p_values.append(p)
    valid_data_subsets.append((x_data, y_data, r))

_, q_values, _, _ = multitest.multipletests(p_values, alpha=0.05, method='fdr_bh')

# ==========================================
# 4. Figure Generation
# ==========================================
fig, axes = plt.subplots(4, 3, figsize=(16, 20), dpi=600)
color_inulin = '#8B008B'

for i, (ax, config) in enumerate(zip(axes.flatten(), plot_configs)):
    x_col, xlabel, y_col, ylabel = config
    x_data, y_data, r = valid_data_subsets[i]
    q = q_values[i]

    line_color, line_alpha = ('black', 1.0) if abs(r) > 0.5 else ('#A0A0A0', 0.5)

    sns.regplot(
        x=x_data, y=y_data, ax=ax,
        scatter_kws={'color': color_inulin, 'edgecolor': 'black', 's': 60, 'linewidths': 1, 'alpha': 0.8},
        line_kws={'color': line_color, 'alpha': line_alpha, 'linewidth': 1.5}
    )

    q_str = "< 0.001" if q < 0.001 else f"= {q:.3f}"

    ax.text(0.95, 0.95, f"$\mathit{{r}}$ = {r:.2f}\n$\mathit{{q}}$ {q_str}",
            transform=ax.transAxes, ha='right', va='top', fontsize=14, fontweight='normal')

    ax.set_xlabel(xlabel, fontsize=14, fontweight='bold')
    if ylabel: ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
    else: ax.set_ylabel('')

    ax.tick_params(axis='both', which='major', labelsize=12)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']: ax.spines[s].set_linewidth(1.5)
    ax.set_box_aspect(1)

plt.tight_layout()
plt.savefig('Supplementary_Figure_S4abcd_InitialFEC.pdf', dpi=600, bbox_inches='tight', transparent=True)
plt.show()